# SAMANVAYA | Constitutional Intelligence Lab
**v0.2.0 · Second educational edition · HoloMathics LLC**

[Read the booklet](https://github.com/Mircus/Samanvaya/blob/main/book/Samanvaya_v0.2.pdf) · [Repository](https://github.com/Mircus/Samanvaya) · [Reference CI](https://github.com/Mircus/Samanvaya/actions/runs/45388976640)

**Intelligence may be distributed. Responsibility cannot disappear.**

Run this notebook from top to bottom on a CPU. No API key, GPU, or paid model is needed. Installation needs network access; the experiments use synthetic facts and simulated effects only. Identities and evidence status are assertions, not authenticated credentials. This is a research prototype, not a production security boundary.

We will distinguish individual authorization, plan coherence, repair and guarded execution.

## 1. Install the exact research baseline
The notebook pins the source revision used in the published CI evidence. The clone is isolated in a temporary directory.

In [ ]:
import sys, subprocess, tempfile
from pathlib import Path

REV = "ac4265d569042d850d4a49794ea3cb775ccd9835"
repo = Path(tempfile.mkdtemp(prefix="samanvaya-lab-")) / "Samanvaya"

def run(*args, **kwargs):
    return subprocess.run(list(args), cwd=repo, check=True, **kwargs)

subprocess.run(["git", "clone", "--quiet", "https://github.com/Mircus/Samanvaya.git", str(repo)], check=True)
run("git", "checkout", "--quiet", REV)
run(sys.executable, "-m", "pip", "install", "-q", "-e", ".")
print("Reference commit:", run("git", "rev-parse", "HEAD", capture_output=True, text=True).stdout.strip())

## 2. Reproduce the official demonstrations
The reference suite compares the action-only baseline to the plan-level checker; it is not a comparison with other vendors.

In [ ]:
run(sys.executable, "examples/land_community_demo.py")
run(sys.executable, "benchmarks/reference_suite.py")

## 3. A bounded mandate
The synthetic trustee authorizes the scout to inspect the site or consider an option, but no broader resources or purposes are granted.

In [ ]:
from datetime import timedelta
from dataclasses import replace
from samanvaya import (
    ActionIntent, AuditLog, ConstitutionConfig, ConstitutionalGateway,
    ConstitutionalPolicy, Effect, Evidence, EvidenceStatus, Mandate, MandateBook, Verdict,
)
from samanvaya.models import utcnow
from samanvaya.coherence import (
    CoherenceAnalyzer, ConstitutionalPlan, FactObligation, FactWitness,
    PlanStep, BeforeObligation, ExclusionObligation,
)
from samanvaya.repair import MinimalRepairExplorer, RepairEdit
from samanvaya.adapters import BoundTool, ConstitutionalToolBroker
from samanvaya.adapters.broker import canonical_args_digest

at = utcnow()
mandates = MandateBook(("human:trustee",))
mandates.issue(Mandate(
    "m-land", "human:trustee", "agent:scout", ("option", "survey"),
    ("land:tuscany",), ("community-feasibility",), (Effect.READ, Effect.WRITE),
    at + timedelta(days=1), max_uses=10,
))
policy = ConstitutionalPolicy(mandates, ConstitutionConfig(("human:trustee",)))
analyzer = CoherenceAnalyzer(policy)
evidence = (Evidence("report:land-registry", "sha256:registry", EvidenceStatus.VERIFIED),)
option = ActionIntent("intent:option", "agent:scout", "m-land", "option",
                      "land:tuscany/site-a", "community-feasibility", Effect.WRITE, evidence)
survey = ActionIntent("intent:survey", "agent:scout", "m-land", "survey",
                      "land:tuscany/site-a", "community-feasibility", Effect.READ)
print("Option:", policy.evaluate(option, at=at).verdict.value)
assert policy.evaluate(option, at=at).verdict is Verdict.ALLOW
outside = replace(survey, intent_id="intent:outside", resource="land:other/site-b")
print("Out-of-scope survey:", policy.evaluate(outside, at=at).verdict.value)
assert policy.evaluate(outside, at=at).verdict is Verdict.DENY

## 4. Locally allowed, globally unknown
The option requires a supported water fact. No such fact exists yet. The individual action is unchanged; the plan verdict is UNKNOWN.

In [ ]:
plan = ConstitutionalPlan("plan:community", (PlanStep("decision", option),), (
    FactObligation("water", "decision", "water_available", ("yes",), trigger_actions=("option",)),
))
assessment = analyzer.assess(plan, at=at)
print(assessment.verdict.value)
for finding in assessment.findings:
    print(finding.rule_id, finding.code, finding.explanation)
assert assessment.verdict is Verdict.UNKNOWN

## 5. Fresh, conflicting and expired facts
Change the witnesses, not the mandate. The word VERIFIED is only a synthetic assertion in this prototype.

In [ ]:
yes = FactWitness("water_available", "yes", "survey:A", "sha256:yes",
                  EvidenceStatus.VERIFIED, at + timedelta(hours=1))
no = replace(yes, value="no", source="survey:B", digest="sha256:no")
stale = replace(yes, valid_until=at - timedelta(seconds=1))
for name, facts, expected in [
    ("supported", (yes,), Verdict.ALLOW),
    ("conflicting", (yes, no), Verdict.DENY),
    ("expired", (stale,), Verdict.UNKNOWN),
]:
    result = analyzer.assess(replace(plan, facts=facts), at=at)
    print(name, result.verdict.value)
    assert result.verdict is expected

## 6. Repair the plan, not the constitution
The operator offers two alternatives: an out-of-scope survey at cost 1 and a valid survey at cost 2. Neither is executed by the search.

In [ ]:
catalog = (
    RepairEdit("outside-survey", "decision", outside, 1, "Invalid resource scope"),
    RepairEdit("water-survey", "decision", survey, 2, "Survey before deciding"),
)
explorer = MinimalRepairExplorer(analyzer)
report = explorer.explore(plan, catalog, at=at)
print("Explored:", report.explored_variants, "/", report.catalog_variants)
for candidate in report.feasible:
    print([edit.edit_id for edit in candidate.edits], candidate.cost, candidate.assessment.verdict.value)
print("Complete:", report.complete, "Scoped minimality:", report.minimal_proven_within_catalog)
assert report.feasible[0].cost == 2 and report.minimal_proven_within_catalog
truncated = explorer.explore(plan, catalog, at=at, max_variants=1)
assert not truncated.complete and not truncated.minimal_proven_within_catalog
print("Truncated search proof:", truncated.minimal_proven_within_catalog)

## 7. Bind the actual arguments and simulate execution
The original proposal cannot execute. After explicit operator selection, bind the revised survey. A changed argument must be refused. No actual land transaction, survey or external call occurs.

In [ ]:
audit_path = Path(tempfile.mkdtemp(prefix="samanvaya-audit-")) / "decisions.jsonl"
gateway = ConstitutionalGateway(policy, AuditLog(audit_path))
args = {"site": "site-a"}
effects = []
original_broker = ConstitutionalToolBroker(gateway, (
    BoundTool("option", lambda _: option, "decision", canonical_args_digest(args)),
), plan, analyzer)
blocked = original_broker.invoke("option", args, lambda: effects.append("UNEXPECTED"))
assert not blocked.executed and not effects
print("Original:", blocked.code)

# Explicit operator selection for this exercise, not automatic acceptance by the model.
chosen = report.feasible[0].plan
broker = ConstitutionalToolBroker(gateway, (
    BoundTool("survey", lambda _: survey, "decision", canonical_args_digest(args)),
), chosen, analyzer)
tampered = broker.invoke("survey", {"site": "site-b"}, lambda: effects.append("UNEXPECTED"))
assert not tampered.executed and not effects
print("Changed arguments:", tampered.code)
allowed = broker.invoke("survey", args, lambda: effects.append("SIMULATED SURVEY") or "ok")
assert allowed.executed and effects == ["SIMULATED SURVEY"]
print("Selected repair:", allowed.code, effects)
run(sys.executable, "-m", "samanvaya", "verify-audit", str(audit_path))

## 8. Run the tests
The optional installed-framework tests skip if LangChain is absent. The reference CI ran them separately with the actual installed package.

In [ ]:
run(sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v")

## 9. Optional: real LangChain framework, fixed model
Set `RUN_LANGCHAIN = True` to install the reference LangChain packages and run the two actual framework integration tests. There are still no model API calls. The test exercises a narrow synchronous middleware path, not secure isolation or every LangGraph execution route.

In [ ]:
import os
RUN_LANGCHAIN = False  # change to True if desired
if RUN_LANGCHAIN:
    run(sys.executable, "-m", "pip", "install", "-q", "-e", ".[langchain]",
        "langchain==1.5.2", "langchain-core==1.8.4")
    run(sys.executable, "-m", "unittest", "tests.test_langchain_real", "-v",
        env={**os.environ, "SAMANVAYA_REQUIRE_LANGCHAIN": "1"})
else:
    print("Optional LangChain experiment not requested.")

## 10. Exercises and research questions

1. Add a second optional step and an `ExclusionObligation` that permits at most one. Show that two individually allowed actions can be denied jointly.
2. Declare a `BeforeObligation` between two steps. Reverse their order and inspect the result. What additional runtime evidence would prove that the prerequisite actually completed?
3. Replace the survey edit with an unauthorized intent. What happens to the feasible set? Is lack of a repair evidence that no conceivable repair exists?
4. Compare JSON digests for reordered keys and for modified values. What additional guarantee would a signed attestation provide?
5. Design a conformance test for a trusted isolated tool service. What prevents a direct SDK bypass?

**Further work:** [security model](https://github.com/Mircus/Samanvaya/blob/main/docs/SECURITY.md) · [non-duplication study](https://github.com/Mircus/Samanvaya/blob/main/docs/DIFFERENTIATOR_AND_NON_DUPLICATION.md) · [experimental protocol](https://github.com/Mircus/Samanvaya/blob/main/docs/EXPERIMENT_PROTOCOL.md).

Do not connect this lab to real privileged services.